Passo 1: Bloco de importação de bibliotecas — cole e use-as no topo do seu notebook dataview.ipynb

In [28]:
#Importar as bibliotecas

import pandas as pd
import numpy as np
import random
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

RF01 – Criar ou Carregar o Dataset de Vendas

O projeto deve usar um dataset de vendas com ao menos as colunas: id | data | cliente | produto | categoria | regiao | quantidade | preco

O dataset deve conter dados intencionalmente sujos: valores None, datas inválidas e espaços extras em strings — para você praticar a limpeza na RF03.

O projeto deverá incluir um dataset de vendas. O estudante pode gerar o dataset sinteticamente no próprio código (recomendado para facilitar a reprodução) ou utilizar um arquivo CSV real. Opção A – Gerar o dataset no código (recomendado):

In [29]:
# Gerar dataset de vendas

def gerar_dataset_vendas(n_registros=1500, seed=42):

    random.seed(seed)
    np.random.seed(seed)

    produtos = ["Notebook","Smartphone","Tablet","Monitor","Teclado","Mouse"]

    precos = {
        "Notebook":3500,
        "Smartphone":2200,
        "Tablet":1800,
        "Monitor":1200,
        "Teclado":250,
        "Mouse":120
    }

    categorias = {
        "Notebook":"Computadores",
        "Smartphone":"Celulares",
        "Tablet":"Celulares",
        "Monitor":"Computadores",
        "Teclado":"Periféricos",
        "Mouse":"Periféricos"
    }

    regioes = ["Sul","Sudeste","Nordeste","Centro-Oeste","Norte"]

    clientes = [f"Cliente_{i:03d}" for i in range(1,31)]

    data_inicio = datetime(2024,1,1)

    dados=[]

    for i in range(n_registros):

        produto = random.choice(produtos)

        dados.append({

            "id":i+1,

            "data":(data_inicio + timedelta(days=random.randint(0,364))).strftime("%Y-%m-%d"),

            "cliente":random.choice(clientes),

            "produto":produto,

            "categoria":categorias[produto],

            "regiao":random.choice(regioes),

            "quantidade":random.randint(1,10),

            "preco":precos[produto]

        })

    df = pd.DataFrame(dados)
    
    return df


In [30]:
# Inserir dados sujos no dataset

df = gerar_dataset_vendas()

# 60 clientes com espaços
idx = np.random.choice(df.index,60,replace=False)
df.loc[idx,"cliente"] = (
" " + df.loc[idx,"cliente"] + " "
)

# 30 regiões com espaços
idx = np.random.choice(df.index,30,replace=False)
df.loc[idx,"regiao"] = (
" " + df.loc[idx,"regiao"]
)

# 5 clientes nulos
idx = np.random.choice(df.index,5,replace=False)
df.loc[idx,"cliente"] = None

# 3 produtos nulos
idx = np.random.choice(df.index,3,replace=False)
df.loc[idx,"produto"] = None

# 10 quantidades nulas
idx = np.random.choice(df.index,10,replace=False)
df.loc[idx,"quantidade"] = None

#Preços nulos
idx = np.random.choice(df.index,5,replace=False)
df.loc[idx,"preco"] = None

#50 datas inválidas
datas_invalidas = [
"31/02/2024",
"99/99/9999",
"abc",
"",
"32/01/2024",
"2024-15-01",
"2024-02-30",
"15-15-2024",
"2024/99/10",
"data_invalida"
]

idx = np.random.choice(df.index,50,replace=False)

for i, linha in enumerate(idx):
    df.loc[linha,"data"] = datas_invalidas[i % len(datas_invalidas)]



# Outliers em preço e quantidade
idx = np.random.choice(df.index, 10, replace=False)
df.loc[idx, "preco"] = np.random.randint(30000, 100000, 10)

idx = np.random.choice(df.index, 20, replace=False)
df.loc[idx, "quantidade"] = np.random.randint(500, 1500, 20)

# Seleciona 50 linhas aleatórias
duplicados = df.sample(n=50, random_state=42)

# Adiciona essas linhas novamente ao DataFrame
df = pd.concat([df, duplicados], ignore_index=True)



print("Dados sujos adicionados com sucesso!")

# Salvar dataset com dados sujos

df.to_csv("../data/raw/vendas_sujo.csv", index=False)

print(df.head())

Dados sujos adicionados com sucesso!
   id        data      cliente     produto     categoria    regiao  \
0   1  2024-02-27  Cliente_001       Mouse   Periféricos  Nordeste   
1   2  2024-03-12  Cliente_024  Smartphone     Celulares       Sul   
2   3  2024-10-29  Cliente_014    Notebook  Computadores       Sul   
3   4  2024-04-21  Cliente_008    Notebook  Computadores     Norte   
4   5  2024-10-14  Cliente_007    Notebook  Computadores     Norte   

   quantidade   preco  
0         4.0   120.0  
1         9.0  2200.0  
2         1.0  3500.0  
3        10.0  3500.0  
4         7.0  3500.0  


In [33]:
#Conferir o dataset sujo


import pandas as pd




# INSPEÇÃO DOS DADOS SUJOS


print("=" * 60)
print("PRIMEIRAS 5 LINHAS")
print("=" * 60)
print(df.head())

print("\n" + "=" * 60)
print("ÚLTIMAS 5 LINHAS")
print("=" * 60)
print(df.tail())

print("\n" + "=" * 60)
print("DIMENSÕES DO DATASET")
print("=" * 60)
print(f"Linhas : {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")

print("\n" + "=" * 60)
print("TIPOS DE DADOS")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("INFORMAÇÕES GERAIS")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("VALORES NULOS")
print("=" * 60)
print(df.isnull().sum())

print("\n" + "=" * 60)
print("REGISTROS DUPLICADOS")
print("=" * 60)
print(df.duplicated().sum())

print("\n" + "=" * 60)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 60)
print(df.describe())

print("\n" + "=" * 60)
print("ESTATÍSTICAS (INCLUINDO COLUNAS DE TEXTO)")
print("=" * 60)
print(df.describe(include="all"))

PRIMEIRAS 5 LINHAS
   id        data      cliente     produto     categoria    regiao  \
0   1  2024-02-27  Cliente_001       Mouse   Periféricos  Nordeste   
1   2  2024-03-12  Cliente_024  Smartphone     Celulares       Sul   
2   3  2024-10-29  Cliente_014    Notebook  Computadores       Sul   
3   4  2024-04-21  Cliente_008    Notebook  Computadores     Norte   
4   5  2024-10-14  Cliente_007    Notebook  Computadores     Norte   

   quantidade   preco  
0         4.0   120.0  
1         9.0  2200.0  
2         1.0  3500.0  
3        10.0  3500.0  
4         7.0  3500.0  

ÚLTIMAS 5 LINHAS
       id        data        cliente   produto     categoria   regiao  \
1545  355  2024-05-26   Cliente_003     Tablet     Celulares      Sul   
1546  366  2024-12-02   Cliente_030     Tablet     Celulares    Norte   
1547  310  2024-10-27   Cliente_018   Notebook  Computadores    Norte   
1548  737  2024-03-09   Cliente_021    Monitor  Computadores  Sudeste   
1549  999  2024-11-30   Cliente_0

RF03 – Limpar e Tratar os Dados O projeto deverá realizar a limpeza dos dados. O estudante deve tratar ao menos os seguintes problemas: Remover ou imputar linhas com valores nulos nas colunas críticas (quantidade, preco_unitario); Remover linhas com datas inválidas; Converter a coluna de data para o tipo datetime; Remover espaços extras em colunas de texto com .str.strip(); Registrar no console quantos registros foram removidos.

In [24]:
# Carregar o dataset

df = pd.read_csv("../data/raw/vendas_sujo.csv")

print("=" * 50)
print("INICIANDO LIMPEZA DOS DADOS")
print("=" * 50)

# Quantidade inicial
registros_iniciais = len(df)


# Remover espaços extras


colunas_texto = [
    "cliente",
    "produto",
    "categoria",
    "regiao"
]

for coluna in colunas_texto:
    df[coluna] = df[coluna].astype(str).str.strip()

# Converter data

df["data"] = pd.to_datetime(df["data"], errors="coerce")


# Remover datas inválidas

antes = len(df)

df = df.dropna(subset=["data"])

datas_removidas = antes - len(df)


# Remover valores nulos


antes = len(df)

df = df.dropna(subset=["cliente","quantidade", "preco", "produto"])

nulos_removidos = antes - len(df)


df = df.drop_duplicates().reset_index(drop=True)
duplicados_removidos = registros_iniciais - len(df) - datas_removidas - nulos_removidos


# Total removido

registros_finais = len(df)

total_removidos = registros_iniciais - registros_finais


# Exibir resumo


print("\nResumo da Limpeza")

print("-" * 30)

print(f"Registros iniciais : {registros_iniciais}")
print(f"Datas inválidas removidas : {datas_removidas}")
print(f"Nulos removidos : {nulos_removidos}")
print(f"Duplicados removidos : {duplicados_removidos}")
print(f"Total removido : {total_removidos}")
print(f"Regitros finais : {registros_finais}")


# Salvar dataset limpo


df.to_csv(
    "../data/processed/v1_com_outliers/vendas_limpo.csv",
    index=False
)

print("\nArquivo salvo com sucesso!")
print("../data/processed/v1_com_outliers/vendas_limpo.csv")

INICIANDO LIMPEZA DOS DADOS

Resumo da Limpeza
------------------------------
Registros iniciais : 1550
Datas inválidas removidas : 53
Nulos removidos : 22
Duplicados removidos : 46
Total removido : 121
Regitros finais : 1429

Arquivo salvo com sucesso!
../data/processed/v1_com_outliers/vendas_limpo.csv


In [34]:
#Conferir o dataset limpo


import pandas as pd

# Carregar o dataset
vendasl = pd.read_csv("../data/processed/v1_com_outliers/vendas_limpo.csv")


# INSPEÇÃO DOS DADOS


print("=" * 60)
print("PRIMEIRAS 5 LINHAS")
print("=" * 60)
print(vendasl.head())

print("\n" + "=" * 60)
print("ÚLTIMAS 5 LINHAS")
print("=" * 60)
print(vendasl.tail())

print("\n" + "=" * 60)
print("DIMENSÕES DO DATASET")
print("=" * 60)
print(f"Linhas : {vendasl.shape[0]}")
print(f"Colunas: {vendasl.shape[1]}")

print("\n" + "=" * 60)
print("TIPOS DE DADOS")
print("=" * 60)
print(vendasl.dtypes)

print("\n" + "=" * 60)
print("INFORMAÇÕES GERAIS")
print("=" * 60)
vendasl.info()

print("\n" + "=" * 60)
print("VALORES NULOS")
print("=" * 60)
print(vendasl.isnull().sum())

print("\n" + "=" * 60)
print("REGISTROS DUPLICADOS")
print("=" * 60)
print(vendasl.duplicated().sum())

print("\n" + "=" * 60)
print("ESTATÍSTICAS DESCRITIVAS")
print("=" * 60)
print(vendasl.describe())

print("\n" + "=" * 60)
print("ESTATÍSTICAS (INCLUINDO COLUNAS DE TEXTO)")
print("=" * 60)
print(vendasl.describe(include="all"))

PRIMEIRAS 5 LINHAS
   id        data      cliente     produto     categoria    regiao  \
0   1  2024-02-27  Cliente_001       Mouse   Periféricos  Nordeste   
1   2  2024-03-12  Cliente_024  Smartphone     Celulares       Sul   
2   3  2024-10-29  Cliente_014    Notebook  Computadores       Sul   
3   4  2024-04-21  Cliente_008    Notebook  Computadores     Norte   
4   5  2024-10-14  Cliente_007    Notebook  Computadores     Norte   

   quantidade   preco  
0         4.0   120.0  
1         9.0  2200.0  
2         1.0  3500.0  
3        10.0  3500.0  
4         7.0  3500.0  

ÚLTIMAS 5 LINHAS
        id        data      cliente     produto     categoria        regiao  \
1424  1495  2024-11-01  Cliente_025       Mouse   Periféricos  Centro-Oeste   
1425  1496  2024-11-26  Cliente_022     Monitor  Computadores           Sul   
1426  1498  2024-12-04  Cliente_005  Smartphone     Celulares  Centro-Oeste   
1427  1499  2024-07-18  Cliente_020      Tablet     Celulares         Norte   
142

RF04 – Detectar e Tratar Outliers (versões v1 e v2) Após a limpeza geral (RF03), o estudante deve trabalhar os outliers das colunas numéricas e gerar duas versões dos dados, salvas em data/processed/: v1_com_outliers/ — os dados limpos da RF03, com os outliers mantidos; v2_outliers_tratado/ — a mesma base da v1, agora com os outliers tratados (removidos ou substituídos). Espera-se que o estudante: detecte os outliers com um método estatístico (IQR ou z-score); gere as limpezas de dados v1 e v2; e registre no notebook quantos outliers foram encontrados. O ponto central é a reutilização: a mesma função de limpeza alimenta ambas as versões.

In [35]:
# Salvar o dataset limpo para a versão 1 (v1) com outliers
df_v1 = vendasl.copy()

df_v1.to_csv(
    "../data/processed/v1_com_outliers/vendas_v1.csv",
    index=False
)

In [39]:
#Utilizar a função tratar_outliers para remover outliers do dataset limpo

import pandas as pd

def tratar_outliers(df, colunas, fator=1.5, metodo="remover"):
    """
    Detecta e trata outliers usando o método IQR.

    Parâmetros:
        df: DataFrame
        colunas: lista de colunas numéricas
        fator: multiplicador do IQR (padrão = 1.5)
        metodo:
            'remover' -> remove as linhas
            'limitar' -> substitui pelos limites (winsorização)
    """

    df = df.copy()

    for col in colunas:

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)

        iqr = q3 - q1

        limite_inferior = q1 - fator * iqr
        limite_superior = q3 + fator * iqr

        outliers = (
            (df[col] < limite_inferior) |
            (df[col] > limite_superior)
        )

        print(f"{col}: {outliers.sum()} outliers encontrados")

        if metodo == "remover":
            df = df[~outliers]

        else:
            df[col] = df[col].clip(
                lower=limite_inferior,
                upper=limite_superior
            )

    return df

In [43]:
#Criação das colunas temporárias para calcular a receita total
df_v1_tmp = df_v1.copy()

df_v1_tmp["receita_total"] = (
    df_v1_tmp["quantidade"] *
    df_v1_tmp["preco"]
)

In [47]:
#Visualizar as primeiras linhas do dataset temporário
print(df_v1_tmp.head())

   id        data      cliente     produto     categoria    regiao  \
0   1  2024-02-27  Cliente_001       Mouse   Periféricos  Nordeste   
1   2  2024-03-12  Cliente_024  Smartphone     Celulares       Sul   
2   3  2024-10-29  Cliente_014    Notebook  Computadores       Sul   
3   4  2024-04-21  Cliente_008    Notebook  Computadores     Norte   
4   5  2024-10-14  Cliente_007    Notebook  Computadores     Norte   

   quantidade   preco  receita_total  
0         4.0   120.0          480.0  
1         9.0  2200.0        19800.0  
2         1.0  3500.0         3500.0  
3        10.0  3500.0        35000.0  
4         7.0  3500.0        24500.0  


In [51]:
#Gerar a versao v2, Aplicar a função tratar_outliers para remover outliers das colunas "quantidade" e "receita_total"

df_v2 = tratar_outliers(

    df_v1_tmp,

    colunas=[
        "quantidade",
        "receita_total"
    ],

    metodo="remover"

)

quantidade: 18 outliers encontrados
receita_total: 60 outliers encontrados


In [ ]:
#Remover a coluna temporária "receita_total" do dataset v2
df_v2.drop(
    columns="receita_total",
    inplace=True
)

In [ ]:
#Comparar o número de linhas entre v1 e v2
print()

print(f"v1 = {len(df_v1)} linhas")

print(f"v2 = {len(df_v2)} linhas")

print(f"Foram removidas {len(df_v1)-len(df_v2)} linhas.")


v1 = 1429 linhas
v2 = 1351 linhas
Foram removidas 78 linhas.


In [ ]:
#Geração do diretório para salvar o dataset v2 tratado

os.makedirs(
    "../data/processed/v2_outliers_tratado",
    exist_ok=True
)

df_v2.to_csv(
    "../data/processed/v2_outliers_tratado/vendas_v2.csv",
    index=False
)

print("v2 salva com sucesso.")

v2 salva com sucesso.


A versão v2 foi escolhida porque, além da limpeza de valores nulos, datas inválidas e espaços extras, os outliers das colunas quantidade e receita_total foram detectados pelo método do Intervalo Interquartil (IQR) e removidos antes da etapa final de análise. Isso reduz a influência de valores extremos nas análises estatísticas e em futuros modelos de aprendizado de máquina.

In [ ]:
#Escolha da versão final do dataset (v2) para salvar como versão final

os.makedirs(
    "../data/final",
    exist_ok=True
)

df_v2.to_csv(
    "../data/final/v2_final.csv",
    index=False
)

print("versão final salva com sucesso.")

versão final salva com sucesso.


RF05 – Criar Colunas Derivadas com Transformações
O projeto deverá criar novas colunas a partir dos dados existentes, enriquecendo o dataset para análise. Deve-se criar ao menos:
receita_total: quantidade * preco_unitario
mes: mês extraído da data (número ou nome)
trimestre: trimestre do ano (Q1, Q2, Q3, Q4)
ano: ano extraído da data
faixa_receita_item: classificação da receita por item usando transformação condicional
